In [1]:
import numpy as np
from veripulse.pulse import run_grape_si, run_grape, run_crab, PulseConfig, PulseResult
from veripulse.gates import rx, rhox, pack_subspace_states, extract_subspace_states, Qobj, operator_to_vector, vector_to_operator
%matplotlib inline

from pathlib import Path

In [2]:
# root path
path_data = Path.cwd().parent/"data"
path_data

PosixPath('/users/home/gustiani/VeriPulse/data')

## Default pulse configuration is based on the paper below 

***from  Frank et al., npj Quantum Information 3, 48 (2017)***
<ul>
    <li>Hilbert space:    2-level (|0⟩, |−1⟩ subspace)</li>
    <li>Hamiltonian:      H = 2π·Δ·Sz + 2π·Ω·[X(t)·Sx + Y(t)·Sy]</li>
    <li>Ω_max:            10 MHz</li>
    <li>T:                75 ns  (π-pulse)</li>
<li>n_tslots:         100</li>
<li>Detuning range:   Δ ∈ [-10Ω, +10Ω] for robustness scan</li>
<li>Decoherence:      optional pure dephasing, T2* ~ 1–3 µs (negligible for single 75 ns pulse)</li>
<li>Target (π):       |0⟩ → |−1⟩</li>
<li>Target (π/2):     U = (I − i·σx)/√2</li>
 <li>AWG timing resolution: ~15ps. The AWG can update the pulse amplitude every 1/65×10⁹ s ≈ 15.4 ps</li>   
<li>AWG amplitude resolution8 bit: The amplitude X(t) ∈ [-1, 1] is discretized into 2⁸ = 256 levels</li>
</ul>


In [4]:
# default pulse setting is based on the npj paper
conf1 = PulseConfig() 
conf1.print()

Parameter                    Value  Description                   
───────────────────────────────────────────────────────────────────────────
omega_drift              1.000e+07  (Hz) Drift/Rabi frequency
T2_star                  2.000e-06  (s) T2-star
drive_error              0.000e+00  [0,1] Misalignment of control on the x- and y-axis
detuning                 0.000e+00  (coefficient) Detuning with respect to omega
evo_time                 7.500e-08  (s) Total evolution time
num_tslots                     100  (int) Number of pulses
amp_lbound              -1.000e+00  (coefficient) Lower bound of pulse amplitude
amp_ubound               1.000e+00  (coefficient) Upper bound of pulse amplitude
awg_resolution           1.500e-11  (ps) AWG resolution
fid_err_targ             1.000e-09  (float) Convergence criteria
max_iter                       500  (int) Cap of iterations
max_wall_time                  120  (s) Cap of compute time
init_pulse_type                RND  (str) First pulse gu

In [5]:
angles= [0, np.pi, np.pi/4, 5*np.pi/4, np.pi/2, 3*np.pi/2, 3*np.pi/4, 7*np.pi/4]

In [6]:
# initial states with a little noise 
err = 0
vrho_init = operator_to_vector(Qobj([[1-err, 0], [0, err]]))
# target states 
rho_targets = [Qobj(rhox(a)) for a in angles]

In [7]:
# GRAPE optimisation part
result_grape = []
for rho_targ in rho_targets:
    res = run_grape(vrho_init, operator_to_vector(rho_targ), config=conf1)
    result_grape.append(res)

In [8]:
pr_grape = PulseResult(
    config=conf1,
    mode="GRAPE",
    results=result_grape,
    rho_targets=rho_targets,
    final_amps=np.stack([r.final_amps for r in result_grape]),
    label="experiment 1",
    state_labels=angles
)

In [9]:
pr_grape.save(path_data/"test_grape.json")

Saved → /users/home/gustiani/VeriPulse/data/test_grape.json


In [29]:
pr_grape.save(path_data/"test_grape.json")

Saved → /users/home/gustiani/VeriPulse/data/test_grape.json


In [6]:
pulse_res = PulseResult.load(path_data/"test_grape.json")

In [8]:
pulse_res.display()


════════════════════════════════════════════════════════════
  experiment 1  [GRAPE]
  detuning=0.000e+00  drive_error=0.000e+00
════════════════════════════════════════════════════════════
         err_HS        err_Uhlmann   fid_compute               termination
───────────────────────────────────────────────────────────────────────────
      5.370e-09          7.329e-05        6               function converged
      4.420e-07          6.650e-04        9               function converged
      6.587e-06          2.570e-03       18               function converged
      6.491e-06          2.405e-03       66               function converged
      5.158e-06          2.274e-03       63               function converged
      7.249e-06          2.696e-03       29               function converged
      4.399e-06          2.099e-03        9               function converged
      2.197e-06          1.483e-03        8               function converged
──────────────────────────────────────────

# 1. Regular CRAB - most commonly used

In [6]:
result_crab = []
for rho_targ in rho_targets:
    res = run_crab(operator_to_vector(rho_init), operator_to_vector(rho_targ), config=conf1)
    result_crab.append(res)

In [46]:
print_fidelities_si(result_crab, rho_targets)

err_Hilbert-Schmidt    err_Uhlmann      fid compute     termination_reason    
───────────────────────────────────────────────────────────────────────────
      4.862e-05          3.813e-04       572       Maximum number of iterations reached
      3.225e-04          1.459e-03       566       Maximum number of iterations reached
      6.942e-02          1.525e-01       568       Maximum number of iterations reached
      1.186e-01          2.772e-01       568       Maximum number of iterations reached
      1.814e-01          4.786e-01       568       Maximum number of iterations reached
      1.758e-01          4.580e-01       567       Maximum number of iterations reached
      7.305e-02          1.594e-01       568       Maximum number of iterations reached
      1.295e-02          2.760e-02       567       Maximum number of iterations reached
────────────────────────────────────────────────────────────
avg:       7.894e-02          1.944e-01       568      
si:       1.714e-01
si_l

# 2. Regular GRAPE

In [7]:
# ── GRAPE ────────────────────────────────────
pr_grape = PulseResult(
    config=conf1,
    mode="GRAPE",
    results=result_grape,
    rho_targets=rho_targets,
    final_amps=np.stack([r.final_amps for r in result_grape]),
    label="grape_det0"
)

NameError: name 'PulseResult' is not defined

In [47]:
print_fidelities_si(result_grape, rho_targets)

err_Hilbert-Schmidt    err_Uhlmann      fid compute     termination_reason    
───────────────────────────────────────────────────────────────────────────
      1.798e-08          1.341e-04        5        function converged
      7.621e-07          8.733e-04        6        function converged
      1.557e-06          1.245e-03       36        function converged
      7.294e-06          2.704e-03       24        function converged
      5.790e-06          2.409e-03       55        function converged
      7.313e-06          2.708e-03       32        function converged
      6.390e-06          2.531e-03        9        function converged
      6.825e-07          8.264e-04        7        function converged
────────────────────────────────────────────────────────────
avg:       3.726e-06          1.679e-03       21       
si:       1.281e-03
si_lb:       3.355e-03


# 3. GRAPE with noise averaging methode

List of results

In [229]:
res8 = []

Pairs of angles that are entirely orthogonal

In [ ]:
angles_pair = [(0, np.pi), (np.pi/4, 5*np.pi/4), (np.pi/2, 3*np.pi/2), (3*np.pi/4, 7*np.pi/4)]

## Packing into 8 states at once

In [ ]:
# packing states 
K = 8
angles_flat = [x for t in angles_pair for x in t]
vRho_init, vRho_target, U_big = pack_subspace_states(
    rotations = [rx(t) for t in angles_flat],
    rho_init = rho_init,
)

In [ ]:
result8_2 = run_grape_si(vRho_init, vRho_target, U_big, lam=400, config=conf1)

In [ ]:
print_fidelities_si_packed(result8_2, angles_flat)
result8_2.termination_reason

In [230]:
result8 = run_grape_si(vRho_init, vRho_target, U_big, lam=1000, config=conf1)

Secret Independence =  0.46506645029982474
fidelity error =  0.14090667250593047
Secret Independence =  0.4628408061980873
fidelity error =  0.12636702897466945
Secret Independence =  0.4423035634625256
fidelity error =  0.1437208547366296
Secret Independence =  0.28664160814212314
fidelity error =  0.1489800327655134
Secret Independence =  0.47558303694881177
fidelity error =  0.10698458180091486
Secret Independence =  0.29376790942614817
fidelity error =  0.1528176507445072
Secret Independence =  0.2756594650733697
fidelity error =  0.15117921220132105
Secret Independence =  0.23072004021554454
fidelity error =  0.15774432625939058
Secret Independence =  0.2879052066539163
fidelity error =  0.16669837340026264
Secret Independence =  0.0891942192950547
fidelity error =  0.1820106643990252
Secret Independence =  0.053170484417800395
fidelity error =  0.18616452587066884
Secret Independence =  0.012218057841816045
fidelity error =  0.1890947260837652
Secret Independence =  0.00534040024

In [1]:
print_fidelities_si_packed(result8, angles_flat)

NameError: name 'print_fidelities_si_packed' is not defined

# Experimenting 

## Packing into 4 states at once

In [68]:
conf2 = PulseConfig(omega_drift = 5e6, # 1-10 MHz
                    num_tslots = 120, # number of pulses
                    T2_star = 3e-6, # 
                    evo_time = 2e-7,
                    drive_error = 0.05,
                    detuning = 0.1, # prop to rabi
                    fid_err_scale_factor = 2,
                    fid_err_targ=1e-14, 
                    max_iter=2000, 
                    max_wall_time=5000) 
print_pulse_config(conf2)

Parameter                      Value Description
─────────────────────────────────────────────
omega_drift                5.000e+06  Hz
T2_star                    3.000e-06  s
drive_error                5.000e-02  
detuning                   1.000e-01  normalised
evo_time                   2.000e-07  s
num_tslots                       120  
amp_lbound                -1.000e+00  
amp_ubound                 1.000e+00  
fid_err_targ               1.000e-14  
max_iter                        2000  
max_wall_time                   5000  s
init_pulse_type                  RND  
fid_params                        {}  
fid_err_scale_factor               2  


### batch 1

In [60]:
angles4_batch1 = [t[0] for t in angles_pair]
angles4_batch2 = [t[1] for t in angles_pair]
print(angles4_batch1, angles4_batch2)
print(angles_pair)

[0, 0.7853981633974483, 1.5707963267948966, 2.356194490192345] [3.141592653589793, 3.9269908169872414, 4.71238898038469, 5.497787143782138]
[(0, 3.141592653589793), (0.7853981633974483, 3.9269908169872414), (1.5707963267948966, 4.71238898038469), (2.356194490192345, 5.497787143782138)]


In [42]:
vRho_init, vRho_target, U_big = pack_subspace_states(
    rotations=[rx(t) for t in angles4_batch1],
    rho_init=rho_init,
)

In [43]:
res4_split1 = run_grape_si(vRho_init, vRho_target, U_big, lam=1000, config=conf2)

Secret Independence =  0.35788481116556636
fidelity error =  0.1670083762853743
Secret Independence =  0.39977520637519093
fidelity error =  0.19643141932792468
Secret Independence =  0.1323286871607312
fidelity error =  0.37498238453454424
Secret Independence =  0.13150470157042274
fidelity error =  0.119356543349962
Secret Independence =  0.42100149069466497
fidelity error =  0.20150221512252958
Secret Independence =  0.044268937101156806
fidelity error =  0.10065881411610128
Secret Independence =  0.21437360432910046
fidelity error =  0.1338207417879751
Secret Independence =  0.010146675738710044
fidelity error =  0.09409009664250569
Secret Independence =  0.0009255974998291179
fidelity error =  0.09003783662445523
Secret Independence =  3.8129991885636494e-06
fidelity error =  0.09006631787374716
Secret Independence =  4.1934999107050594e-07
fidelity error =  0.09004756470070112
Secret Independence =  2.2392989061078285e-07
fidelity error =  0.0899905773028028
Secret Independence =

In [44]:
print_fidelities_si_packed(res4_split1, angles4_batch1)

Err fid Uhlmann
───────────────
    4.814e-03
    5.332e-03
    5.193e-03
    5.302e-03
───────────────
si :  3.826e-03
si_lb:     3.883e-02


## batch 2

In [75]:
vRho_init, vRho_target, U_big = pack_subspace_states(
    rotations=[rx(t) for t in angles4_batch2],
    rho_init=rho_init,
)
res4_split2 = run_grape_si(vRho_init, vRho_target, U_big, lam=10000, config=conf2)

Secret Independence =  0.3068478080327743
fidelity error =  0.27145707440626493
Secret Independence =  0.2678684882246669
fidelity error =  0.25186853817784505
Secret Independence =  0.09827853365307243
fidelity error =  0.2748615123930893
Secret Independence =  0.04201750777061716
fidelity error =  0.27928827283713975
Secret Independence =  0.042630210111364544
fidelity error =  0.2872265731456637
Secret Independence =  0.010613950948097224
fidelity error =  0.2848517261201699
Secret Independence =  0.024705996523112977
fidelity error =  0.29892730625495484
Secret Independence =  0.003414078436891616
fidelity error =  0.2906527048674672
Secret Independence =  0.00032153749082683674
fidelity error =  0.2916478207507957
Secret Independence =  2.1007234875023545e-05
fidelity error =  0.29140986955776216
Secret Independence =  1.021668210636141e-06
fidelity error =  0.2911750373465673
Secret Independence =  7.772656817821761e-07
fidelity error =  0.2911360638456758
Secret Independence =  

In [77]:
print_fidelities_si_packed(res4_split2, angles4_batch2)

Err fid Uhlmann
───────────────
    1.185e-02
    1.167e-02
    1.215e-02
    1.201e-02
───────────────
si :  2.194e-04
si_lb:     1.161e-01


In [55]:
res4 = [res4_split1, res4_split2]
angles4 = [*angles4_batch1, *angles4_batch2]
print_fidelities_si_packed_list(res4, angles4)

Err fid Uhlmann
───────────────
    4.814e-03
    5.332e-03
    5.193e-03
    5.302e-03
───────────────
si :  5.802e-03
si_lb:     2.397e-02
